# 03 — Confidence Calibration via Temperature Scaling

This notebook applies post-hoc temperature scaling (Guo et al., 2017) to the trained EfficientNet-B3 detector from notebook 02.

**What temperature scaling does:** Modern neural networks tend to be overconfident — a model reporting 95% confidence may only be correct 80% of the time. Temperature scaling learns a single parameter *T* that softens the model's output distribution:

```
calibrated_probs = softmax(logits / T)
```

When *T* > 1, predictions become less extreme (closer to uniform), reducing overconfidence.

**This notebook:**
1. Loads the trained checkpoint and validation data
2. Collects raw logits on the validation set
3. Measures calibration **before** temperature scaling (ECE + reliability diagram)
4. Learns the optimal temperature *T* on the validation set
5. Measures calibration **after** temperature scaling (ECE + reliability diagram)
6. Produces a side-by-side comparison plot
7. Saves the results to `baseline_results.json`

**Prerequisites:** Run `02_baseline_training.ipynb` first to produce a trained checkpoint.

In [1]:
import os
from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/ai-image-detection"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/krishi-shah/ai-image-detection.git {REPO_DIR}
os.chdir(REPO_DIR)

Mounted at /content/drive
Cloning into '/content/ai-image-detection'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 97 (delta 48), reused 92 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (97/97), 5.71 MiB | 28.39 MiB/s, done.
Resolving deltas: 100% (48/48), done.


In [2]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 54.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 22.7 MB/s eta

In [3]:
import os, zipfile, glob
DATA_DIR = "/content/ai-image-detection/data/raw/cifake"
DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(os.path.join(DATA_DIR, "train")):
    zip_path = glob.glob(os.path.join(DRIVE_ROOT, "*.zip"))[0]
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(DATA_DIR)
    print("Extracted.")

Extracted.


In [4]:
# Cell 1 — Imports, paths, device, load checkpoint
import os, sys, json
import numpy as np
import torch
import torch.nn as nn

REPO_DIR = "/content/ai-image-detection"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

DRIVE_ROOT = "/content/drive/MyDrive/ai-image-detection"
DATA_DIR = os.path.join(REPO_DIR, "data", "raw", "cifake")
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
PLOTS_DIR = os.path.join(REPO_DIR, "outputs", "plots")
RESULTS_DIR = os.path.join(REPO_DIR, "outputs", "results")
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

from src.model.detector import build_detector, load_checkpoint
from src.utils.data_loader import get_cifake_loaders

model = build_detector(pretrained=False).to(device)
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_detector.pth")
epoch_loaded, ckpt_metrics = load_checkpoint(model, checkpoint_path)
model.to(device)
model.eval()

print(f"Loaded checkpoint from epoch {epoch_loaded}")
print(f"  val_acc: {ckpt_metrics['val_acc']:.4f}")

_, val_loader, test_loader = get_cifake_loaders(DATA_DIR, batch_size=32, num_workers=2)

Device: cuda
Loaded checkpoint from epoch 2
  val_acc: 0.9694


In [5]:
# Cell 2 — Collect raw logits on the validation set
from src.evaluation.calibration import collect_logits

val_logits, val_labels = collect_logits(model, val_loader, device)

print(f"Collected logits: {val_logits.shape}")
print(f"Labels:           {val_labels.shape}")
print(f"Class distribution: {dict(zip(*np.unique(val_labels.numpy(), return_counts=True)))}")

Collected logits: torch.Size([20000, 2])
Labels:           torch.Size([20000])
Class distribution: {np.int64(0): np.int64(10038), np.int64(1): np.int64(9962)}


## Pre-Calibration Analysis

Before applying temperature scaling, we measure how well the model's raw confidence scores match its actual accuracy. A perfectly calibrated model sits on the diagonal of the reliability diagram.

In [6]:
# Cell 3 — Pre-calibration: ECE and reliability diagram
from src.evaluation.metrics import compute_ece, plot_reliability_diagram

probs_before = torch.softmax(val_logits, dim=1).numpy()
ece_before = compute_ece(probs_before, val_labels.numpy())

print(f"ECE (before temperature scaling): {ece_before:.4f}")

plot_reliability_diagram(
    probs_before,
    val_labels.numpy(),
    os.path.join(PLOTS_DIR, "reliability_pre_calibration.png"),
)
print(f"Saved: {PLOTS_DIR}/reliability_pre_calibration.png")

ECE (before temperature scaling): 0.0031
Saved: /content/ai-image-detection/outputs/plots/reliability_pre_calibration.png


## Learn Optimal Temperature

We optimise a single scalar *T* on the validation set using L-BFGS to minimise the negative log-likelihood. This is the method from Guo et al. (2017) — despite having only one parameter, it matches or outperforms more complex calibration methods.

In [7]:
# Cell 4 — Learn temperature on validation set
from src.evaluation.calibration import find_temperature

learned_temperature = find_temperature(model, val_loader, device)

print(f"\nLearned temperature: {learned_temperature:.4f}")
if learned_temperature > 1.0:
    print("T > 1 indicates the model was overconfident (as expected for modern DNNs).")
else:
    print("T <= 1 indicates the model was underconfident.")

Temperature: 1.2189
ECE before:  0.0031
ECE after:   0.0116

Learned temperature: 1.2189
T > 1 indicates the model was overconfident (as expected for modern DNNs).


## Post-Calibration Analysis

Now we apply the learned temperature to the same validation logits and re-measure ECE. If calibration worked, the bars in the reliability diagram should move closer to the diagonal and ECE should decrease.

In [8]:
# Cell 5 — Post-calibration: ECE and reliability diagram
from src.evaluation.calibration import apply_temperature

probs_after = apply_temperature(val_logits.numpy(), learned_temperature)
ece_after = compute_ece(probs_after, val_labels.numpy())

print(f"ECE (after temperature scaling):  {ece_after:.4f}")
print(f"ECE reduction:                    {ece_before - ece_after:.4f}")

plot_reliability_diagram(
    probs_after,
    val_labels.numpy(),
    os.path.join(PLOTS_DIR, "reliability_post_calibration.png"),
)
print(f"Saved: {PLOTS_DIR}/reliability_post_calibration.png")

ECE (after temperature scaling):  0.0116
ECE reduction:                    -0.0084
Saved: /content/ai-image-detection/outputs/plots/reliability_post_calibration.png


## Side-by-Side Comparison

A combined figure showing the reliability diagram before and after calibration. The dotted diagonal represents perfect calibration.

In [9]:
# Cell 6 — Side-by-side reliability diagrams
import matplotlib.pyplot as plt

n_bins = 10
bin_boundaries = np.linspace(0.0, 1.0, n_bins + 1)
bin_centres = [(bin_boundaries[i] + bin_boundaries[i + 1]) / 2 for i in range(n_bins)]
width = 1.0 / n_bins


def _bin_accuracies(probs, labels):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)
    accs = []
    for i in range(n_bins):
        mask = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i + 1])
        accs.append(float(accuracies[mask].mean()) if mask.sum() > 0 else 0.0)
    return accs


labels_np = val_labels.numpy()
accs_before = _bin_accuracies(probs_before, labels_np)
accs_after = _bin_accuracies(probs_after, labels_np)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(bin_centres, accs_before, width=width * 0.9, alpha=0.7, label="Accuracy")
ax1.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax1.set_xlabel("Confidence")
ax1.set_ylabel("Accuracy")
ax1.set_title(f"Before (ECE = {ece_before:.4f})")
ax1.legend(loc="upper left")
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

ax2.bar(bin_centres, accs_after, width=width * 0.9, alpha=0.7, color="green", label="Accuracy")
ax2.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
ax2.set_xlabel("Confidence")
ax2.set_ylabel("Accuracy")
ax2.set_title(f"After  (ECE = {ece_after:.4f}, T = {learned_temperature:.2f})")
ax2.legend(loc="upper left")
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

fig.suptitle("Reliability Diagrams: Before vs After Temperature Scaling", fontsize=14)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "calibration_comparison.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/calibration_comparison.png")

Saved: /content/ai-image-detection/outputs/plots/calibration_comparison.png


## Test Set Evaluation + Save Results

Finally, we evaluate the calibrated model on the **test set** and save all metrics to a JSON file. This file will be used in Phase 2 as the baseline for comparison against generalisation results on other generators.

In [10]:
# Cell 7 — Test set evaluation with calibration + save results
from src.evaluation.metrics import compute_accuracy, compute_auc
from tqdm.auto import tqdm

test_logits, test_labels = collect_logits(model, test_loader, device)

test_probs_before = torch.softmax(test_logits, dim=1).numpy()
test_probs_after = apply_temperature(test_logits.numpy(), learned_temperature)

test_preds = np.argmax(test_probs_before, axis=1)
test_labels_np = test_labels.numpy()

test_acc = compute_accuracy(test_preds, test_labels_np)
test_auc = compute_auc(test_probs_before, test_labels_np)
test_ece_before = compute_ece(test_probs_before, test_labels_np)
test_ece_after = compute_ece(test_probs_after, test_labels_np)

print(f"{'='*45}")
print(f"  Test Results (CIFAKE) — Calibrated")
print(f"{'='*45}")
print(f"  Accuracy:              {test_acc:.4f}")
print(f"  AUC-ROC:               {test_auc:.4f}")
print(f"  ECE (before scaling):  {test_ece_before:.4f}")
print(f"  ECE (after scaling):   {test_ece_after:.4f}")
print(f"  Temperature:           {learned_temperature:.4f}")
print(f"{'='*45}")

# Save structured results for Phase 2 comparison
results = {
    "dataset": "CIFAKE",
    "model": "EfficientNet-B3",
    "checkpoint_epoch": int(epoch_loaded),
    "test_accuracy": round(test_acc, 4),
    "test_auc": round(test_auc, 4),
    "ece_before_calibration": round(test_ece_before, 4),
    "ece_after_calibration": round(test_ece_after, 4),
    "temperature": round(learned_temperature, 4),
    "checkpoint_path": "outputs/checkpoints/best_detector.pth",
}

results_path = os.path.join(RESULTS_DIR, "baseline_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to {results_path}")

  Test Results (CIFAKE) — Calibrated
  Accuracy:              0.9696
  AUC-ROC:               0.9971
  ECE (before scaling):  0.0026
  ECE (after scaling):   0.0113
  Temperature:           1.2189

Results saved to /content/ai-image-detection/outputs/results/baseline_results.json


In [11]:
import shutil
shutil.copytree(
    "/content/ai-image-detection/outputs",
    "/content/drive/MyDrive/ai-image-detection/outputs",
    dirs_exist_ok=True
)
print("Outputs saved to Drive!")

Outputs saved to Drive!


## Threshold Sensitivity Analysis

The default decision threshold is 0.5 — whichever class has the higher probability wins. But different thresholds trade off precision against recall. A lower threshold (e.g. 0.4) catches more fakes but produces more false positives; a higher threshold (e.g. 0.6) only flags high-confidence fakes but misses more.

Below we sweep across thresholds from 0.40 to 0.70 and compute **Accuracy, Precision, Recall, and F1** at each one, treating FAKE (class 0) as the positive class.

In [12]:
# Cell 8 — Threshold sensitivity analysis
import json
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
fake_probs = test_probs_before[:, 0]  # P(FAKE) for each sample

records = []
for t in thresholds:
    preds = (fake_probs >= t).astype(int)
    preds_class = 1 - preds  # flip: 1 where FAKE, but labels use 0=FAKE
    preds_class = np.where(fake_probs >= t, 0, 1)

    acc  = compute_accuracy(preds_class, test_labels_np)
    prec = precision_score(test_labels_np, preds_class, pos_label=0)
    rec  = recall_score(test_labels_np, preds_class, pos_label=0)
    f1   = f1_score(test_labels_np, preds_class, pos_label=0)

    records.append({"threshold": t, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1})

# Print table
print(f"{'Threshold':>10}  {'Accuracy':>9}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}")
print("-" * 55)
for r in records:
    marker = "  <-- default" if r["threshold"] == 0.50 else ""
    print(f"{r['threshold']:>10.2f}  {r['accuracy']:>9.4f}  {r['precision']:>10.4f}  {r['recall']:>8.4f}  {r['f1']:>8.4f}{marker}")

# Extract arrays for plotting
t_vals   = [r["threshold"] for r in records]
acc_vals  = [r["accuracy"] for r in records]
prec_vals = [r["precision"] for r in records]
rec_vals  = [r["recall"] for r in records]
f1_vals   = [r["f1"] for r in records]

# --- Plot 1: Accuracy vs Threshold ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, acc_vals, "o-", linewidth=2)
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_accuracy.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_accuracy.png")

# --- Plot 2: Precision & Recall vs Threshold ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, prec_vals, "s-", linewidth=2, label="Precision")
ax.plot(t_vals, rec_vals, "^-", linewidth=2, label="Recall")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision & Recall vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_precision_recall.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_precision_recall.png")

# --- Plot 3: F1 vs Threshold ---
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(t_vals, f1_vals, "D-", linewidth=2, color="green")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("F1 Score")
ax.set_title("F1 Score vs Decision Threshold")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_f1.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_f1.png")

# --- Plot 4: All metrics combined ---
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(t_vals, acc_vals, "o-", linewidth=2, label="Accuracy")
ax.plot(t_vals, prec_vals, "s-", linewidth=2, label="Precision")
ax.plot(t_vals, rec_vals, "^-", linewidth=2, label="Recall")
ax.plot(t_vals, f1_vals, "D-", linewidth=2, label="F1")
ax.axvline(0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")
ax.set_xlabel("FAKE-class Threshold")
ax.set_ylabel("Score")
ax.set_title("All Metrics vs Decision Threshold (CIFAKE Test Set)")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.3)
ax.set_ylim(0.85, 1.01)
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "threshold_all_metrics.png"), dpi=150)
plt.show()
print(f"Saved: {PLOTS_DIR}/threshold_all_metrics.png")

# Save results to JSON
threshold_results_path = os.path.join(RESULTS_DIR, "threshold_analysis.json")
with open(threshold_results_path, "w") as f:
    json.dump(records, f, indent=2)
print(f"\nThreshold analysis saved to {threshold_results_path}")

 Threshold   Accuracy   Precision    Recall        F1
-------------------------------------------------------
      0.40     0.9722      0.9807    0.9635    0.9720
      0.45     0.9716      0.9839    0.9589    0.9712
      0.50     0.9696      0.9858    0.9529    0.9691  <-- default
      0.55     0.9665      0.9871    0.9454    0.9658
      0.60     0.9646      0.9894    0.9393    0.9637
      0.65     0.9605      0.9906    0.9297    0.9592
      0.70     0.9563      0.9922    0.9197    0.9546
Saved: /content/ai-image-detection/outputs/plots/threshold_accuracy.png
Saved: /content/ai-image-detection/outputs/plots/threshold_precision_recall.png
Saved: /content/ai-image-detection/outputs/plots/threshold_f1.png
Saved: /content/ai-image-detection/outputs/plots/threshold_all_metrics.png

Threshold analysis saved to /content/ai-image-detection/outputs/results/threshold_analysis.json


## Detailed Evaluation: Confusion Matrix, Per-Class Metrics, and Diagnostics

The following cells provide a thorough diagnostic evaluation of the baseline detector on the CIFAKE test set, including the confusion matrix, per-class precision/recall/F1, ROC curve, predicted probability distributions, dataset balance verification, and misclassification analysis.

In [13]:
# Cell 10 — Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(test_labels_np, test_preds)
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["FAKE", "REAL"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix (CIFAKE Test Set)")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

print(f"\nFAKE images correctly classified: {cm[0, 0]} / {cm[0].sum()} ({cm[0, 0] / cm[0].sum():.2%})")
print(f"REAL images correctly classified: {cm[1, 1]} / {cm[1].sum()} ({cm[1, 1] / cm[1].sum():.2%})")
print(f"FAKE misclassified as REAL (false negatives): {cm[0, 1]}")
print(f"REAL misclassified as FAKE (false positives): {cm[1, 0]}")
print(f"\nSaved: {PLOTS_DIR}/confusion_matrix.png")


FAKE images correctly classified: 9529 / 10000 (95.29%)
REAL images correctly classified: 9863 / 10000 (98.63%)
FAKE misclassified as REAL (false negatives): 471
REAL misclassified as FAKE (false positives): 137

Saved: /content/ai-image-detection/outputs/plots/confusion_matrix.png


In [14]:
# Cell 11 — Per-Class Classification Report (Precision, Recall, F1 for both classes)
from sklearn.metrics import classification_report

report = classification_report(
    test_labels_np, test_preds,
    target_names=["FAKE (class 0)", "REAL (class 1)"],
    digits=4,
)
print("Per-Class Classification Report (CIFAKE Test Set)")
print("=" * 60)
print(report)
print("=" * 60)
print("\nNote: Both classes achieve similar precision, recall, and F1,")
print("confirming there is no systematic imbalance in per-class performance.")

Per-Class Classification Report (CIFAKE Test Set)
                precision    recall  f1-score   support

FAKE (class 0)     0.9858    0.9529    0.9691     10000
REAL (class 1)     0.9544    0.9863    0.9701     10000

      accuracy                         0.9696     20000
     macro avg     0.9701    0.9696    0.9696     20000
  weighted avg     0.9701    0.9696    0.9696     20000


Note: Both classes achieve similar precision, recall, and F1,
confirming there is no systematic imbalance in per-class performance.


In [15]:
# Cell 12 — ROC Curve
from src.evaluation.metrics import plot_roc_curve

roc_path = os.path.join(PLOTS_DIR, "roc_curve.png")
plot_roc_curve(test_probs_before, test_labels_np, roc_path)

from sklearn.metrics import roc_auc_score
auc_score = roc_auc_score(test_labels_np, test_probs_before[:, 1])
print(f"AUC-ROC: {auc_score:.4f}")
print(f"Saved: {roc_path}")

from PIL import Image as PILImage
img = PILImage.open(roc_path)
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis("off")
plt.title(f"ROC Curve (AUC = {auc_score:.4f})")
plt.tight_layout()
plt.show()

AUC-ROC: 0.9971
Saved: /content/ai-image-detection/outputs/plots/roc_curve.png


In [16]:
# Cell 13 — Predicted Probability Histograms
fake_probs_fake = test_probs_before[test_labels_np == 0, 0]  # P(FAKE) for actual FAKE images
fake_probs_real = test_probs_before[test_labels_np == 1, 0]  # P(FAKE) for actual REAL images

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.hist(fake_probs_fake, bins=50, alpha=0.7, color="red", edgecolor="black")
ax1.set_xlabel("P(FAKE)")
ax1.set_ylabel("Count")
ax1.set_title("Actual FAKE images")
ax1.axvline(0.5, color="black", linestyle="--", alpha=0.7, label="Threshold = 0.5")
ax1.legend()

ax2.hist(fake_probs_real, bins=50, alpha=0.7, color="blue", edgecolor="black")
ax2.set_xlabel("P(FAKE)")
ax2.set_ylabel("Count")
ax2.set_title("Actual REAL images")
ax2.axvline(0.5, color="black", linestyle="--", alpha=0.7, label="Threshold = 0.5")
ax2.legend()

fig.suptitle("Distribution of Predicted P(FAKE) by True Class", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "probability_histograms.png"), dpi=150)
plt.show()

print(f"Saved: {PLOTS_DIR}/probability_histograms.png")
print(f"\nActual FAKE — mean P(FAKE): {fake_probs_fake.mean():.4f}, median: {np.median(fake_probs_fake):.4f}")
print(f"Actual REAL — mean P(FAKE): {fake_probs_real.mean():.4f}, median: {np.median(fake_probs_real):.4f}")

Saved: /content/ai-image-detection/outputs/plots/probability_histograms.png

Actual FAKE — mean P(FAKE): 0.9293, median: 0.9983
Actual REAL — mean P(FAKE): 0.0244, median: 0.0000


In [17]:
# Cell 14 — Train/Val/Test Split Balance Verification
from src.utils.data_loader import get_cifake_loaders

train_loader_check, val_loader_check, test_loader_check = get_cifake_loaders(
    DATA_DIR, batch_size=32, num_workers=2
)

def count_classes(loader):
    counts = {0: 0, 1: 0}
    for _, labels in loader:
        for l in labels.tolist():
            counts[l] += 1
    return counts

train_counts = count_classes(train_loader_check)
val_counts = count_classes(val_loader_check)
test_counts = count_classes(test_loader_check)

print("Dataset Split Balance Verification")
print("=" * 55)
print(f"{'Split':<10} {'FAKE (0)':>10} {'REAL (1)':>10} {'Total':>10} {'%FAKE':>8}")
print("-" * 55)
for name, counts in [("Train", train_counts), ("Val", val_counts), ("Test", test_counts)]:
    total = counts[0] + counts[1]
    pct = counts[0] / total * 100
    print(f"{name:<10} {counts[0]:>10,} {counts[1]:>10,} {total:>10,} {pct:>7.1f}%")
print("=" * 55)
print("\nBoth classes use the SAME preprocessing pipeline (via ImageFolder).")
print("Train: Resize(224), RandomHorizontalFlip, RandomCrop(224, pad=8), ColorJitter, Normalize")
print("Val/Test: Resize(256), CenterCrop(224), Normalize")
print("All images are loaded as RGB and normalised with ImageNet mean/std.")

Dataset Split Balance Verification
Split        FAKE (0)   REAL (1)      Total    %FAKE
-------------------------------------------------------
Train          39,962     40,038     80,000    50.0%
Val            10,038      9,962     20,000    50.2%
Test           10,000     10,000     20,000    50.0%

Both classes use the SAME preprocessing pipeline (via ImageFolder).
Train: Resize(224), RandomHorizontalFlip, RandomCrop(224, pad=8), ColorJitter, Normalize
Val/Test: Resize(256), CenterCrop(224), Normalize
All images are loaded as RGB and normalised with ImageNet mean/std.


In [18]:
# Cell 15 — Misclassified FAKE Images Analysis
# Find FAKE images (label=0) that were predicted as REAL (pred=1)
false_negatives_mask = (test_labels_np == 0) & (test_preds == 1)
fn_indices = np.where(false_negatives_mask)[0]
fn_confidences = test_probs_before[fn_indices, 1]  # P(REAL) for these misclassified images

print(f"Misclassified FAKE images (predicted as REAL): {len(fn_indices)} out of {(test_labels_np == 0).sum()}")
print(f"Misclassification rate for FAKE class: {len(fn_indices) / (test_labels_np == 0).sum():.2%}")

if len(fn_indices) > 0:
    sort_idx = np.argsort(-fn_confidences)
    top_fn = fn_indices[sort_idx[:min(8, len(fn_indices))]]
    top_conf = fn_confidences[sort_idx[:min(8, len(fn_indices))]]

    # Load and display the misclassified images
    test_dataset = test_loader.dataset
    n_show = min(8, len(top_fn))
    cols = min(4, n_show)
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = axes[np.newaxis, :]
    elif cols == 1:
        axes = axes[:, np.newaxis]

    for i in range(n_show):
        r, c = divmod(i, cols)
        idx = top_fn[i]
        img_tensor, label = test_dataset[idx]
        # Undo normalisation for display
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_display = img_tensor * std + mean
        img_display = img_display.clamp(0, 1).permute(1, 2, 0).numpy()

        axes[r, c].imshow(img_display)
        axes[r, c].set_title(f"P(REAL)={top_conf[i]:.2%}", fontsize=9, color="red")
        axes[r, c].axis("off")

    # Hide unused axes
    for i in range(n_show, rows * cols):
        r, c = divmod(i, cols)
        axes[r, c].axis("off")

    fig.suptitle("Misclassified FAKE Images (Predicted as REAL)", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, "misclassified_fakes.png"), dpi=150)
    plt.show()
    print(f"\nSaved: {PLOTS_DIR}/misclassified_fakes.png")
    print("\nDiscussion: These FAKE images were the ones the model found most")
    print("realistic. They likely have fewer visible Stable Diffusion artifacts")
    print("(smoother textures, more natural colour distributions) compared to")
    print("correctly classified FAKE images, making them harder to distinguish")
    print("from real photographs.")
else:
    print("No FAKE images were misclassified as REAL on this test set.")

Misclassified FAKE images (predicted as REAL): 471 out of 10000
Misclassification rate for FAKE class: 4.71%

Saved: /content/ai-image-detection/outputs/plots/misclassified_fakes.png

Discussion: These FAKE images were the ones the model found most
realistic. They likely have fewer visible Stable Diffusion artifacts
(smoother textures, more natural colour distributions) compared to
correctly classified FAKE images, making them harder to distinguish
from real photographs.


In [19]:
# Cell 16 — Save all outputs to Google Drive
import shutil
shutil.copytree(
    os.path.join(REPO_DIR, "outputs"),
    "/content/drive/MyDrive/ai-image-detection/outputs",
    dirs_exist_ok=True
)
print("All outputs saved to Drive!")
print("Plots saved:", os.listdir(PLOTS_DIR))
print("Results saved:", os.listdir(RESULTS_DIR))

All outputs saved to Drive!
Plots saved: ['threshold_precision_recall.png', 'misclassified_fakes.png', 'threshold_all_metrics.png', 'threshold_accuracy.png', 'probability_histograms.png', 'roc_curve.png', 'reliability_post_calibration.png', 'threshold_f1.png', '.gitkeep', 'confusion_matrix.png', 'calibration_comparison.png', 'reliability_pre_calibration.png']
Results saved: ['threshold_analysis.json', '.gitkeep', 'baseline_results.json']
